In [0]:
# Databricks notebook source

# ============================================================
# PROYECTO: EACP Automotive Data Lakehouse
# CAPA: Bronze
# NOTEBOOK: 01_eacp_bronze_ingestion
# ============================================================

from pyspark.sql.functions import (
    current_timestamp,
    lit,
    sha2,
    concat_ws,
    col
)

# ------------------------------------------------------------
# 1. PARÁMETROS
# ------------------------------------------------------------

# Cambia este valor por el nombre real de tu Storage Account
storage_account = "steacpauto01"

raw_container = "raw"
raw_folder = "automobile/input"

catalog_name = "catalog"

bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

automobile_file = (
    f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/"
    f"{raw_folder}/Automobile_data.csv"
)

manufacturers_file = (
    f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/"
    f"{raw_folder}/EACP_Automobile_Manufacturers.csv"
)

automobile_table = (
    f"{catalog_name}.{bronze_schema}.automobile_raw"
)

manufacturers_table = (
    f"{catalog_name}.{bronze_schema}."
    f"automobile_manufacturers_raw"
)


# ------------------------------------------------------------
# 3. FUNCIÓN PARA NORMALIZAR NOMBRES
# ------------------------------------------------------------

def normalize_columns(dataframe):
    """
    Convierte nombres como fuel-type y city-mpg
    en fuel_type y city_mpg.
    """

    for original_column in dataframe.columns:

        normalized_column = (
            original_column
            .strip()
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
            .replace("/", "_")
        )

        if original_column != normalized_column:
            dataframe = dataframe.withColumnRenamed(
                original_column,
                normalized_column
            )

    return dataframe

# ------------------------------------------------------------
# 4. LEER AUTOMOBILE_DATA.CSV
# ------------------------------------------------------------

df_automobile_source = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .option("encoding", "UTF-8")
    .load(automobile_file)
    .select(
        "*",

        col("_metadata.file_path").alias(
            "source_file"
        ),

        col("_metadata.file_name").alias(
            "source_file_name"
        ),

        col("_metadata.file_size").alias(
            "source_file_size"
        ),

        col("_metadata.file_modification_time").alias(
            "source_file_modification_time"
        )
    )
)

df_automobile_source = normalize_columns(
    df_automobile_source
)

# Excluir columnas de metadata del cálculo del hash
automobile_business_columns = [
    column_name
    for column_name in df_automobile_source.columns
    if column_name not in [
        "source_file",
        "source_file_name",
        "source_file_size",
        "source_file_modification_time"
    ]
]

df_automobile_bronze = (
    df_automobile_source
    .withColumn(
        "source_system",
        lit("ADLS_GEN2")
    )
    .withColumn(
        "source_dataset",
        lit("Automobile_data.csv")
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
    .withColumn(
        "record_hash",
        sha2(
            concat_ws(
                "||",
                *[
                    col(column_name).cast("string")
                    for column_name
                    in automobile_business_columns
                ]
            ),
            256
        )
    )
)

print("Automobile_data.csv leído correctamente.")
print(
    "Registros:",
    df_automobile_bronze.count()
)

# ------------------------------------------------------------
# 5. LEER EACP_AUTOMOBILE_MANUFACTURERS.CSV
# ------------------------------------------------------------

df_manufacturers_source = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .option("encoding", "UTF-8")
    .load(manufacturers_file)
    .select(
        "*",

        col("_metadata.file_path").alias(
            "source_file"
        ),

        col("_metadata.file_name").alias(
            "source_file_name"
        ),

        col("_metadata.file_size").alias(
            "source_file_size"
        ),

        col("_metadata.file_modification_time").alias(
            "source_file_modification_time"
        )
    )
)

df_manufacturers_source = normalize_columns(
    df_manufacturers_source
)

manufacturer_business_columns = [
    column_name
    for column_name in df_manufacturers_source.columns
    if column_name not in [
        "source_file",
        "source_file_name",
        "source_file_size",
        "source_file_modification_time"
    ]
]

df_manufacturers_bronze = (
    df_manufacturers_source
    .withColumn(
        "source_system",
        lit("ADLS_GEN2")
    )
    .withColumn(
        "source_dataset",
        lit(
            "EACP_Automobile_Manufacturers.csv"
        )
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
    .withColumn(
        "record_hash",
        sha2(
            concat_ws(
                "||",
                *[
                    col(column_name).cast("string")
                    for column_name
                    in manufacturer_business_columns
                ]
            ),
            256
        )
    )
)

print(
    "EACP_Automobile_Manufacturers.csv leído correctamente."
)

print(
    "Registros:",
    df_manufacturers_bronze.count()
)

# ------------------------------------------------------------
# 6. CREAR TABLAS DELTA BRONZE
# ------------------------------------------------------------

(
    df_automobile_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(automobile_table)
)

print(
    f"Tabla creada: {automobile_table}"
)

(
    df_manufacturers_bronze.write    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{bronze_schema}.automobile_manufacturers_raw")
)


Automobile_data.csv leído correctamente.
Registros: 205
EACP_Automobile_Manufacturers.csv leído correctamente.
Registros: 22
Tabla creada: catalog.bronze.automobile_raw
